In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

root = Path(r"C:\Dev\Bachelorarbeit\results\accounting\runs\synth_final")
files = sorted(root.rglob("metrics_per_scenario.csv"))
len(files), files[0]

dfs = []
for f in files:
    df = pd.read_csv(f)

    rel = f.relative_to(root)
    parts = rel.parts
    # Erwartet: <config>/<run_name>/E2/_report/metrics_per_scenario.csv
    config = parts[0] if len(parts) > 0 else "unknown"
    run_name = parts[1] if len(parts) > 1 else "run"
    df["config_label"] = f"{config}/{run_name}"
    dfs.append(df)

all_df = pd.concat(dfs, ignore_index=True)

# KPI-Spalten automatisch (nur numerische)
drop_cols = {"fold", "scenario", "test_dir", "run", "config_label"}
kpi_cols = [c for c in all_df.columns if c not in drop_cols and pd.api.types.is_numeric_dtype(all_df[c])]

# Mittelwert über folds => pro Szenario + Config genau eine Zeile
agg = (all_df.groupby(["scenario", "config_label"], as_index=False)[kpi_cols]
            .mean(numeric_only=True))

tables = {}
for sc in sorted(agg["scenario"].unique()):
    tab = agg[agg["scenario"] == sc].set_index("config_label")[kpi_cols].copy()
    tab = tab.replace([np.inf, -np.inf], np.nan)

    # optional: sortieren nach ex_cum_return (falls vorhanden)
    if "ex_cum_return" in tab.columns:
        tab = tab.sort_values("ex_cum_return", ascending=False)

    tables[sc] = tab

tables.keys()


dict_keys(['bear_1y', 'side_highvol_1y', 'side_lowvol_1y'])

In [2]:
out_dir = root / "_tables"
out_dir.mkdir(exist_ok=True)

for sc, tab in tables.items():
    tab.to_csv(out_dir / f"synth_table_{sc}.csv")

    latex = tab.to_latex(
        escape=True,
        float_format=lambda x: f"{x:.4f}" if pd.notna(x) else ""
    )
    wrapper = (
        "\\begin{table}[ht]\n\\centering\n\\footnotesize\n"
        f"\\caption{{Synth-Ergebnisse pro Konfiguration im Szenario \\texttt{{{sc}}}}}\n"
        f"\\label{{tab:synth-{sc}}}\n"
        "\\resizebox{\\textwidth}{!}{%\n"
        + latex +
        "}\n\\end{table}\n"
    )
    (out_dir / f"synth_table_{sc}.tex").write_text(wrapper, encoding="utf-8")

out_dir


WindowsPath('C:/Dev/Bachelorarbeit/results/accounting/runs/synth_final/_tables')